In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv('messy_customer_data.csv')
df = raw.copy()

def quality_report(d):
    return {
        'rows': len(d),
        'total_nulls': int(d.isnull().sum().sum()),
        'duplicate_rows': int(d.duplicated().sum()),
        'correct_dtypes': int(sum([
            pd.api.types.is_string_dtype(d['Customer_ID']) or d['Customer_ID'].dtype == object,
            pd.api.types.is_datetime64_any_dtype(d['Signup_Date']),
            pd.api.types.is_float_dtype(d['Purchase_Amount']),
        ])),
    }

before = quality_report(raw)
print('Shape:', raw.shape)
print('\nNulls per column:\n', raw.isnull().sum())
print('\nData types:\n', raw.dtypes)
print('\nDuplicate rows:', before['duplicate_rows'])
raw.head(10)

Shape: (315, 8)

Nulls per column:
 Customer_ID         0
Name                8
Gender              0
Age                15
City               13
Signup_Date         0
Purchase_Amount    14
Rating             10
dtype: int64

Data types:
 Customer_ID          int64
Name                object
Gender              object
Age                float64
City                object
Signup_Date         object
Purchase_Amount     object
Rating             float64
dtype: object

Duplicate rows: 15


,Customer_ID,Name,Gender,Age,City,Signup_Date,Purchase_Amount,Rating
0,1001,Vikram,Male,53.0,Mumbai,2023-06-14,"$1,138.82",1.0
1,1002,Rahul,F,29.0,Pune,20/04/2023,604.40,2.0
2,1003,Vikram,F,54.0,Nagpur,"Jun 25, 2022",1084.60,4.0
3,1004,Karan,Male,55.0,MUMBAI,2023-02-13,1441.36,4.0
4,1005,Sneha,female,49.0,Nashik,23/09/2022,$835.08,5.0
5,1006,Karan,FEMALE,250.0,MUMBAI,"Oct 22, 2022",425.14,3.0
6,1007,Anjali,FEMALE,32.0,Mumbai,2022-03-20,1252.69,1.0
7,1008,Pooja,male,51.0,Nagpur,09/05/2022,587.76,3.0
8,1009,Sneha,Male,24.0,pune,"Aug 29, 2022",$774.50,4.0
9,1010,Rohan,M,47.0,Pune,2022-07-24,651.14,1.0


In [2]:
print('Gender values  :', raw['Gender'].unique())
print('City values    :', raw['City'].unique())
print('Age min / max  :', raw['Age'].min(), '/', raw['Age'].max())
raw['Purchase_Amount'].head(8).tolist()

Gender values  : [' Male' 'F' 'female' 'FEMALE ' 'male' 'Male' 'M' 'Female']
City values    : ['Mumbai ' 'Pune' 'Nagpur' 'MUMBAI' ' Nashik' 'pune' 'delhi' nan 'Delhi']
Age min / max  : -5.0 / 999.0


['$1,138.82',
 '604.40',
 '1084.60',
 '1441.36',
 '$835.08',
 '425.14',
 '1252.69',
 '587.76']

In [4]:
df['Gender'] = df['Gender'].str.strip().str.lower().map(
    {'m': 'Male', 'male': 'Male', 'f': 'Female', 'female': 'Female'})
df['City'] = df['City'].str.strip().str.title()
df['Name'] = df['Name'].str.strip().str.title()
print(df['Gender'].unique(), df['City'].unique())

['Male' 'Female'] ['Mumbai' 'Pune' 'Nagpur' 'Nashik' 'Delhi' nan]


In [5]:
df['Customer_ID'] = df['Customer_ID'].astype(str)

def parse_date(x):
    if pd.isnull(x):
        return pd.NaT
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%b %d, %Y'):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df['Signup_Date'] = df['Signup_Date'].apply(parse_date)
df['Purchase_Amount'] = (df['Purchase_Amount'].astype(str)
                         .str.replace(r'[$,]', '', regex=True)
                         .replace('nan', np.nan).astype(float))
df.dtypes

Customer_ID                object
Name                       object
Gender                     object
Age                       float64
City                       object
Signup_Date        datetime64[ns]
Purchase_Amount           float64
Rating                    float64
dtype: object

In [6]:
dups = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f'Duplicate rows removed: {dups}')
print('Rows remaining:', len(df))

Duplicate rows removed: 15
Rows remaining: 300


In [7]:
invalid_age = ((df['Age'] < 0) | (df['Age'] > 100)).sum()
df.loc[(df['Age'] < 0) | (df['Age'] > 100), 'Age'] = np.nan
print('Invalid ages set to NaN:', invalid_age)

Invalid ages set to NaN: 3


In [8]:
print('Nulls before:\n', df.isnull().sum(), '\n')

for col in ['Age', 'Purchase_Amount', 'Rating']:
    df[col] = df[col].fillna(df[col].median())
for col in ['Gender', 'City']:
    df[col] = df[col].fillna(df[col].mode()[0])
df['Name'] = df['Name'].fillna('Unknown')

rows_before = len(df)
df = df.dropna(subset=['Signup_Date']).reset_index(drop=True)
print('Rows dropped (missing date):', rows_before - len(df))
print('\nNulls after:\n', df.isnull().sum())

Nulls before:
 Customer_ID         0
Name                8
Gender              0
Age                18
City               12
Signup_Date         0
Purchase_Amount    14
Rating             10
dtype: int64 

Rows dropped (missing date): 0

Nulls after:
 Customer_ID        0
Name               0
Gender             0
Age                0
City               0
Signup_Date        0
Purchase_Amount    0
Rating             0
dtype: int64


In [9]:
def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ['Age', 'Purchase_Amount', 'Rating']:
    lo, hi = iqr_bounds(df[col])
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f'{col:16s} bounds=({lo:.1f}, {hi:.1f})  outliers={n_out}')

Age              bounds=(-5.1, 87.9)  outliers=0
Purchase_Amount  bounds=(-632.1, 2062.1)  outliers=16
Rating           bounds=(-1.0, 7.0)  outliers=0


In [10]:
lo, hi = iqr_bounds(df['Purchase_Amount'])
df['Purchase_Amount'] = df['Purchase_Amount'].clip(lower=max(lo, 0), upper=hi)
print('Purchase_Amount max after capping:', df['Purchase_Amount'].max())

Purchase_Amount max after capping: 2062.06125


In [11]:
after = quality_report(df)
summary = pd.DataFrame({'Before': before, 'After': after})
summary.loc['rows'] = [len(raw), len(df)]
summary

,Before,After
rows,315,300
total_nulls,60,0
duplicate_rows,15,0
correct_dtypes,0,3


In [12]:
df.info()
df.describe(include='all').T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Customer_ID      300 non-null    object        
 1   Name             300 non-null    object        
 2   Gender           300 non-null    object        
 3   Age              300 non-null    float64       
 4   City             300 non-null    object        
 5   Signup_Date      300 non-null    datetime64[ns]
 6   Purchase_Amount  300 non-null    float64       
 7   Rating           300 non-null    float64       
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 18.9+ KB


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Customer_ID,300,300,1001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Name,300,11,Karan,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,300,2,Female,155,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,300.0,NaN,NaN,NaN,41.31,18.0,29.75,43.0,53.0,64.0,13.06482
City,300,5,Pune,85,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Signup_Date,300,NaN,NaN,NaN,2022-12-01 10:19:12,2022-01-03 00:00:00,2022-06-22 00:00:00,2022-10-31 12:00:00,2023-05-31 06:00:00,2023-12-01 00:00:00,NaN
Purchase_Amount,300.0,NaN,NaN,NaN,759.6471,43.86,378.23,640.825,1051.7625,2062.06125,501.939024
Rating,300.0,NaN,NaN,NaN,2.96,1.0,2.0,3.0,4.0,5.0,1.323217


In [13]:
df.to_csv('cleaned_customer_data.csv', index=False)
print('Saved cleaned_customer_data.csv with shape', df.shape)

Saved cleaned_customer_data.csv with shape (300, 8)
